# Boat detector v6 — YOLOv8s at 960 pixels, unfrozen backbone

Start a new run from `yolov8s.pt`, with no frozen backbone, using `boat_v4s_frozen_v2_bundle.zip`. The historical bundle has 3,944 training and 1,241 validation images (5,185 total). This experiment differs from `train_v6.py`, which uses YOLO11m.

Choose a GPU under **Runtime > Change runtime type**, upload the bundle to Google Drive, and update `ZIP_PATH` below. Adjust batch size to available GPU memory.


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set ZIP_PATH to the bundle you uploaded. Extraction resets /content/work.
ZIP_PATH = '/content/drive/MyDrive/trainFreeze/boat_v4s_frozen_v2_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work


In [ ]:
!pip install -q ultralytics==8.4.138


In [ ]:
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- before ---')
print(content)

import yaml
settings = yaml.safe_load(content)
settings['path'] = '/content/work/yolo_dataset_v4'
new_content = yaml.safe_dump(settings, sort_keys=False)
yaml_path.write_text(new_content)
print('--- after ---')
print(yaml_path.read_text())


In [ ]:
# Start this experiment with the settings below.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=960,
    device=0,
    batch=32,
    patience=40,
    project='/content/work/runs_boat_yolo',
    name='boat_v6_960_5k',
)


In [ ]:
# Resume from the latest checkpoint; retain optimizer and scheduler state.
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v6_960_5k/weights/last.pt')
# results = model.train(resume=True)


In [ ]:
!mkdir -p /content/drive/MyDrive/boat_v6_960_5k_results
!cp -r /content/work/runs_boat_yolo/boat_v6_960_5k /content/drive/MyDrive/boat_v6_960_5k_results/
print('Copied to: Google Drive > boat_v6_960_5k_results > boat_v6_960_5k')


## Retrieve and evaluate the trained checkpoint

Download the result folder from the Google Drive destination printed above. Keep `weights/best.pt`, `weights/last.pt`, `args.yaml`, and `results.csv` together so checkpoint provenance is available. Pass the exact `best.pt` path to `run.py --yolo-weights` and evaluate it on independently reviewed recordings.

Colab sessions can disconnect. Keep recent checkpoints in Drive and resume from the latest `last.pt`; do not restart from the initial bundle and assume it contains newer training progress. Session duration and training speed are not guaranteed.
